# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [1]:
! pip install -q schedule pytest requests beautifulsoup4  # установка библиотек, если ещё не

In [2]:
# Библиотеки, которые могут вам понадобиться
import json
import logging
import re
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta
from pathlib import Path
from urllib.parse import urljoin

import requests
import schedule
from bs4 import BeautifulSoup

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
logger = logging.getLogger("books_scraper")

BASE_URL = "http://books.toscrape.com/"
CATALOGUE_PAGE_URL = BASE_URL + "catalogue/page-{}.html"
DEFAULT_OUTPUT = Path("..") / "artifacts" / "books_data.txt"
REQUEST_TIMEOUT_SEC = 15
DEFAULT_WORKERS = 10
SCHEDULER_CHECK_INTERVAL_SEC = 60
DEFAULT_RUN_TIME = "19:00"
RATING_WORDS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [ ]:
def fetch_soup(url: str, session: requests.Session | None = None) -> BeautifulSoup:
    """Скачать страницу и вернуть её разобранное HTML-дерево.

    Args:
        url: адрес страницы.
        session: необязательная сессия ``requests`` для переиспользования
            соединений; если не передана, используется ``requests.get``.

    Returns:
        Объект ``BeautifulSoup`` со страницей.

    Raises:
        requests.HTTPError: если сервер вернул код ошибки (4xx/5xx).
        requests.RequestException: при сетевых ошибках.
    """
    client = session or requests
    response = client.get(url, timeout=REQUEST_TIMEOUT_SEC)
    response.raise_for_status()
    # Берём байты, а не response.text: сайт не указывает кодировку,
    # и requests иначе ломает символ фунта «£».
    return BeautifulSoup(response.content, "html.parser")

def parse_price(text: str) -> float:
    """Преобразовать строку цены вида ``'£51.77'`` в число ``51.77``."""
    return float(re.sub(r"[^\d.]", "", text))

def parse_availability(text: str) -> int:
    """Достать количество экземпляров из строки ``'In stock (22 available)'``.

    Returns:
        Количество книг в наличии, либо 0, если число в строке не найдено.
    """
    match = re.search(r"\d+", text)
    return int(match.group()) if match else 0

def parse_book_page(soup: BeautifulSoup, book_url: str = "") -> dict:
    """Извлечь данные о книге из уже скачанной страницы товара.

    Args:
        soup: HTML-дерево страницы книги.
        book_url: адрес страницы (сохраняется в результат для справки).

    Returns:
        Словарь с ключами ``url``, ``title``, ``price``, ``rating``,
        ``available``, ``description``, ``category`` и ``product_info``
        (все строки таблицы Product Information как ``{заголовок: значение}``).
    """
    main = soup.select_one("div.product_main")
    rating_tag = main.select_one("p.star-rating")
    rating_word = next(
        (cls for cls in rating_tag["class"] if cls in RATING_WORDS), None
    )
    description_tag = soup.select_one("#product_description + p")
    breadcrumbs = soup.select("ul.breadcrumb li a")
    product_info = {
        row.th.get_text(strip=True): row.td.get_text(strip=True)
        for row in soup.select("table.table-striped tr")
    }

    return {
        "url": book_url,
        "title": main.h1.get_text(strip=True),
        "price": parse_price(main.select_one("p.price_color").get_text()),
        "rating": RATING_WORDS.get(rating_word, 0),
        "available": parse_availability(
            main.select_one("p.availability").get_text()
        ),
        "description": (
            description_tag.get_text(strip=True) if description_tag else ""
        ),
        "category": breadcrumbs[-1].get_text(strip=True) if breadcrumbs else "",
        "product_info": product_info,
    }

def get_book_data(book_url: str, session: requests.Session | None = None) -> dict:
    """Собрать всю информацию об одной книге по ссылке на её страницу.

    Args:
        book_url: полный адрес страницы книги, например
            ``http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html``.
        session: необязательная сессия ``requests``.

    Returns:
        Словарь с названием, ценой (float, в фунтах), рейтингом (1–5),
        количеством в наличии, описанием, категорией и таблицей
        Product Information. Формат - см. ``parse_book_page``.
    """
    return parse_book_page(fetch_soup(book_url, session), book_url)

In [4]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'url': 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html',
 'title': 'A Light in the Attic',
 'price': 51.77,
 'rating': 3,
 'available': 22,
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, 

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [ ]:
def get_book_links(page_soup: BeautifulSoup, page_url: str) -> list[str]:
    """Вернуть абсолютные ссылки на все книги со страницы каталога.

    Args:
        page_soup: HTML-дерево страницы каталога.
        page_url: адрес этой страницы (нужен для превращения
            относительных ссылок в абсолютные).

    Returns:
        Список URL страниц книг в порядке их следования на странице.
    """
    return [
        urljoin(page_url, link["href"])
        for link in page_soup.select("article.product_pod h3 a")
    ]

def collect_all_book_links(session: requests.Session) -> list[str]:
    """Пройти по страницам каталога ``page-1``, ``page-2``, ... и собрать ссылки.

    Обход идет, пока на странице есть кнопка «next», поэтому число страниц
    не зашито в код.

    Args:
        session: сессия ``requests``.

    Returns:
        Список ссылок на все книги каталога.
    """
    links = []
    page_number = 1
    while True:
        page_url = CATALOGUE_PAGE_URL.format(page_number)
        page_soup = fetch_soup(page_url, session)
        links = links + get_book_links(page_soup, page_url)
        logger.debug("Страница %d: всего ссылок %d", page_number, len(links))
        if page_soup.select_one("li.next a") is None:
            return links
        page_number += 1

def save_books(books: list[dict], output_path: Path) -> None:
    """Сохранить список книг в текстовый файл.

    Args:
        books: список словарей, полученных из ``get_book_data``.
        output_path: путь к файлу; недостающие папки будут созданы.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as file:
        for book in books:
            file.write(json.dumps(book, ensure_ascii=False) + "\n")
    logger.info("Сохранено %d книг в %s", len(books), output_path)

def scrape_books(
    is_save: bool = False,
    output_path: Path | str = DEFAULT_OUTPUT,
    max_workers: int = DEFAULT_WORKERS,
) -> list[dict]:
    """Собрать данные обо всех книгах каталога Books to Scrape.

    Сначала обходит все страницы каталога и собирает ссылки на книги,
    затем параллельно скачивает страницы книг через ``get_book_data``.

    Args:
        is_save: если ``True``, результат сохраняется в ``output_path``;
            если ``False``, шаг сохранения пропускается.
        output_path: путь к файлу для сохранения
            (по умолчанию ``artifacts/books_data.txt`` рядом со скриптом).
        max_workers: число потоков для параллельной загрузки книг.

    Returns:
        Список словарей с данными о книгах (порядок как в каталоге).
    """
    with requests.Session() as session:
        links = collect_all_book_links(session)
        with ThreadPoolExecutor(max_workers=max_workers) as pool:
            books = list(pool.map(lambda url: get_book_data(url, session), links))

    if is_save:
        save_books(books, Path(output_path))
    return books

In [6]:
# Проверка работоспособности функции
res = scrape_books(is_save=True)
print(type(res), len(res))
print("Уникальных названий:", len({book["title"] for book in res}))
print("Первая книга:", res[0]["title"], res[0]["price"], res[0]["rating"])
print("Файл:", DEFAULT_OUTPUT.resolve().name, "строк:",
      DEFAULT_OUTPUT.read_text(encoding="utf-8").count("\n"))

2026-11-16 11:14:05,039 INFO Сохранено 1000 книг в ../artifacts/books_data.txt


<class 'list'> 1000
Уникальных названий: 999
Первая книга: A Light in the Attic 51.77 3
Файл: books_data.txt строк: 1000


## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [7]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ

def scheduled_job(output_path: Path | str = DEFAULT_OUTPUT) -> None:
    """Задача для планировщика: собрать все книги и сохранить в файл.

    Ошибки сети логируются, а не прерывают планировщик — следующая
    попытка будет в следующий запланированный момент.
    """
    logger.info("Запуск планового сбора данных")
    try:
        books = scrape_books(is_save=True, output_path=output_path)
    except requests.RequestException:
        logger.exception("Сбор данных не удался")
        return
    logger.info("Плановый сбор завершён: %d книг", len(books))

def run_scheduler(
    run_at: str = DEFAULT_RUN_TIME,
    check_interval: int = SCHEDULER_CHECK_INTERVAL_SEC,
    output_path: Path | str = DEFAULT_OUTPUT,
) -> None:
    """Запускать сбор данных каждый день в заданное время (бесконечный цикл).

    Чтобы не нагружать систему, наступление времени проверяется
    не постоянно, а раз в ``check_interval`` секунд.

    Args:
        run_at: время запуска в формате ``"ЧЧ:ММ"``.
        check_interval: пауза между проверками расписания, в секундах.
        output_path: куда сохранять результат.
    """
    schedule.every().day.at(run_at).do(scheduled_job, output_path=output_path)
    logger.info("Планировщик запущен: ежедневно в %s", run_at)
    while True:
        schedule.run_pending()
        time.sleep(check_interval)


# Боевой запуск (бесконечный цикл, каждый день в 19:00) — в ноутбуке не запускаем,
# в проекте он стартует командой `python scraper.py --at 19:00`:
# run_scheduler("19:00")

# Локальная проверка: ставим задачу на ближайшую минуту, ждём ОДИН запуск и выходим.
DEMO_CHECK_INTERVAL_SEC = 5
demo_time = (datetime.now() + timedelta(minutes=1)).strftime("%H:%M")
schedule.clear()
demo_job = schedule.every().day.at(demo_time).do(scheduled_job)
print("Задача запланирована на", demo_time, "| следующий запуск:", demo_job.next_run)

runs_before = demo_job.last_run
while demo_job.last_run == runs_before:
    schedule.run_pending()
    time.sleep(DEMO_CHECK_INTERVAL_SEC)

print("Задача выполнена в", demo_job.last_run.strftime("%H:%M:%S"),
      "| следующий запуск:", demo_job.next_run)
schedule.clear()

# КОНЕЦ ВАШЕГО РЕШЕНИЯ

Задача запланирована на 11:15 | следующий запуск: 2026-11-16 11:15:00


2026-11-16 11:15:00,098 INFO Запуск планового сбора данных


2026-11-16 11:15:37,098 INFO Сохранено 1000 книг в ../artifacts/books_data.txt


2026-11-16 11:15:37,099 INFO Плановый сбор завершён: 1000 книг


Задача выполнена в 11:15:37 | следующий запуск: 2026-11-17 11:15:00


## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [8]:
# Ячейка для демонстрации работоспособности
# Сам код находится в отдельном скрипте tests/test_scraper.py
! cd .. && pytest -v tests/test_scraper.py

============================= test session starts ==============================
platform darwin -- Python 3.14.3, pytest-9.2.0, pluggy-1.7.0 -- /Users/valentina/Desktop/books_scraper/venv/bin/python3.14
cachedir: .pytest_cache
rootdir: /Users/valentina/Desktop/books_scraper
configfile: pytest.ini
plugins: platformdirs-4.14.5, anyio-4.16.0
collecting ... 


collected 7 items                                                              



tests/test_scraper.py::test_parse_book_page_returns_expected_fields 

PASSED [ 14%]


tests/test_scraper.py::test_parse_availability[In stock (22 available)-22] PASSED [ 28%]
tests/test_scraper.py::test_parse_availability[Out of stock-0] PASSED    [ 42%]
tests/test_scraper.py::test_get_book_links_makes_absolute_urls PASSED    [ 57%]
tests/test_scraper.py::test_scrape_books_saves_file_only_when_flag_set PASSED [ 71%]
tests/test_scraper.py::test_get_book_data_real_page 

PASSED               [ 85%]
tests/test_scraper.py::test_catalogue_contains_all_books 

PASSED          [100%]

============================== 7 passed in 14.61s ==============================


## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
#мой проект на гитхаб https://github.com/Tina-03/books_scraper  ```

## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

`Ответ: нужно было написать парсер для сайта, сбрать оттуда информацию, запустить проект на гитхабе.`

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

`Ответ: с помощью requests получала HTML страничку, через BeautifulSoup поиск нужных элементов`

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

`Ответ: у ИИ спрашивала чуть подробнее про залив проекта на гитхаб`

**4. Что осталось непонятным** (честный список, он поможет на консультации):

`Ответ: ...`

**5. Самооценка по критериям задания**:

`Ответ: ...`

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

`Ответ: ...`

> Напоминание: рефлексия пишется своими словами, без помощи ИИ.

